# High-performance and parallel computing for AI - Practical 6: Multi-CPU and multi-GPU computing in JAX - JAX sharding

IMPORTANT
=========

* When using a GPU we need to set some environment variables (see below). If you get some weird error restart the kernel and rerun the code below.
* For these practicals we will be using a different `conda environment`. When opening a notebook or a terminal make sure you are using the **hpc4ai Kernel**!!!
* It's fine if you do not finish everything.

Before you start (and before running any other GPU code on the servers) please run the following code, which attempts to limit the maximum GPU memory usage and picks two L40s GPUs at random, excluding the Quadro GPUs. **Please only run the code below once every time you restart the kernel!**

**NOTE:** Here we are setting the `JAX_NUM_CPU_DEVICES` environment variable to tell JAX we want it to see more than one CPU.

**NOTE:** Here we are also defining some helper functions for later. Ignore them.

In [1]:
import os

# JAX-specific environment variables
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.249" # restrict GPU memory preallocation to a fourth of the total
os.environ["JAX_OPTIMIZATION_LEVEL"]="O1"
os.environ["JAX_NUM_CPU_DEVICES"] = "6" # JAX will see 6 CPUs (if available)

## On goliat we have FIVE GPUs so here we pick two of those at random
## so that we do not overload the system.
## The way we do it is by figuring out the GPU UUIDs and then setting
## The CUDA_VISIBLE_DEVICES environment variable.
## Note: this is useful for other libraries as well (e.g., Jax, PyTorch, TF) in multi-GPU servers.

# To get these UUIDs you need to run nvidia-smi -q on the command line
quadro_UUIDs = ["GPU-4efa947b-abbd-7c6e-84f5-61241d34bb4b",
                "GPU-5eb524b0-2b1b-fe98-e6ed-b8fb5185e993"]

L40s_UUIDs = ["GPU-7bba1f33-03d2-016b-d42e-ced83c3ac243",
              "GPU-179d068a-3bea-91d7-1a8c-7017f55d6298",
              "GPU-ae634859-dd49-de46-9182-195639405eaa"]

import random

a, b = random.sample(range(3), 2) # picks two numbers between 0 and 2 at random

# Picks an L40s and a Quadro GPU at random. The others will be invisible to JAX
# NOTE: this only works if the environment variable is set BEFORE JAX is first imported.
os.environ["CUDA_VISIBLE_DEVICES"] = L40s_UUIDs[a] + "," + L40s_UUIDs[b]

## JAX (or whatever other software) will only see these GPUs.

################################## HELPER FUNCTIONS FOR DEBUGGING - IGNORE THEM #############################

import html
import jax
import jax.numpy as jnp
import ipywidgets as widgets
from IPython.display import display, HTML
import shutil
import time
import numpy as np
from collections import Counter

gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6 and len(gpus) == 2

def print_sharding_info(x):
    left = widgets.Output(layout=widgets.Layout(
        width='50%',
        border='1px solid #ddd',
        padding='8px'
    ))
    right = widgets.Output(layout=widgets.Layout(
        width='50%',
        border='1px solid #ddd',
        padding='8px'
    ))

    print("\n**Debug info:**\n")
    print("jax.typeof(x): ", jax.typeof(x))
    print("\nx.sharding: ", x.sharding, "\n")

    with left:
        print("jax.debug.visualize_array_sharding(x)\n")
        try: jax.debug.visualize_array_sharding(x)
        except (NotImplementedError,ValueError): print("This visualisation option is not implemented for this type of array!")

    with right:
        lines = []
        lines.append("Using x.addressable_shards:\n")
        lines.append(f"Complete array:\n{x}\n")
        for s in x.addressable_shards:
            lines.append(f"{s.device}\n{s.data}\n")
        text = "\n".join(lines)
        display(HTML(f"<pre style='margin:0; white-space:pre-wrap'>{html.escape(text)}</pre>"))

    display(widgets.HBox([left, right], layout=widgets.Layout(width='100%')))

x = jnp.arange(6 * 4.).reshape(6, 4)
print_sharding_info(x)

COLLECTIVE_PATTERNS = {
    "all-reduce": ("all-reduce", "all_reduce"),
    "all-gather": ("all-gather", "all_gather"),
    "all-to-all": ("all-to-all", "all_to_all"),
    "reduce-scatter": ("reduce-scatter", "reduce_scatter"),
}

def count_collectives(jit_fn, *args):
    """Return (total_count, counts_by_type) for collectives in compiled HLO text."""
    hlo = jit_fn.lower(*args).compile().as_text()
    counts = Counter()

    for line in hlo.splitlines():
        line = line.strip().lower()
        for name, patterns in COLLECTIVE_PATTERNS.items():
            if any(p in line for p in patterns):
                counts[name] += 1
                break

    return sum(counts.values()), dict(counts)


def report(jit_fn, *args, label=""):
    total, counts = count_collectives(jit_fn, *args)

    prefix = f"[{label}] " if label else ""
    print(f"{prefix}collectives: {total}")

    if counts:
        parts = [f"{name}={counts[name]}" for name in sorted(counts)]
        print(f"{prefix}types: " + ", ".join(parts))
    else:
        print(f"{prefix}types: none")

def time_fn(fn, *args, warmup=3, repeats=10):
    """Time a JIT'd function. Returns the median time in ms."""
    for _ in range(warmup):
        out = fn(*args)
    jax.block_until_ready(out)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = fn(*args)
        jax.block_until_ready(out)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


**Debug info:**

jax.typeof(x):  float32[6,4]

x.sharding:  SingleDeviceSharding(device=CudaDevice(id=0), memory_kind=device) 



## Exercise 1 - Sharded matmul with automatic sharding

### Part a)

Create a `(2, 3)` mesh formed of 6 CPUs with axes names `"data"` and `"model"` respectively and axes set to `Automatic`. Let $N = 4$, $K = 12$, $M = 6$. Then define the following matrices (using `jnp.arange` and then reshaping):

- `A` of shape `(N, K)` sharded with spec `P("data", None)`.
- `B` of shape `(K, M)` sharded with spec `P(None, "model")`.

The product `C = A @ B` has shape `(N, M)`. Create a function called `matmul` computing the matrix product and JIT-it.

Predict what sharding JAX will infer for `C` and whether any communication is required (advice: use `jax.debug.visualize_array_sharding` to visually inspect the sharding of $A$ and $B$). Check with `jax.debug.visualize_array_sharding(C)` that you were right.

Run the following command, which reveals how many collective communication operations (i.e., all-reduce, all-gather, all-to-all, reduce-scatter) are being used by JAX to compute the matmul.

```python
report(jax.jit(matmul), A, B, label="no-comm matmul")
```
Deduce that JAX has automatically chosen the output sharding to minimise communication.

### Part b)

Repeat the excercise, but shard $A$ and $B$ as follows:

- `A` sharded with spec `P("data", "model")`.
- `B` sharded with spec `P("model", None)`.

Will the matmul trigger any collective communication?

Now, repeat with the shardings

- `A` sharded with spec `P("data", "model")`.
- `B` sharded with spec `P("model", "data")`.

Will this trigger more or less communication?

### Part c)

Take the sharding specs from the last case in Part b) and force the output $C$ to be sharded according to `P('data', 'model')`. In how many different ways you can enforce this? Use the `report` helper function again to check how the communication patterns changed.

Which one of all the previous sharding strategies will be faster in your opinion? Time each option (multiply the array sizes by $100$ and remember to use `jax.block_until_ready`). The difference will be there, albeit not highly significant in this case, why? Hint: think about how the number of floating point operations for the matmul VS the memory to be moved scale with the array sizes.

## Solution

In [2]:
Auto = jax.sharding.AxisType.Auto
mesh = jax.make_mesh((2,3), ("data", "model"), (Auto, Auto), devices=cpus)
jax.set_mesh(mesh)

timing_flag = False # set to True for timing computations
fac = 100 if timing_flag else 1
N, K, M = 4*fac, 12*fac, 6*fac 

A_h = jnp.arange(N*K).reshape((N,K))
B_h = jnp.arange(K*M).reshape((K,M))

A = jax.device_put(A_h, jax.P("data", None))
B = jax.device_put(B_h, jax.P(None, "model"))

@jax.jit
def matmul(a, b):
    return a @ b

def timematmul(a,b):
    c = matmul(a,b)
    c = jax.block_until_ready(c)
    return c

C = matmul(A, B)
jax.block_until_ready(C)

if timing_flag:
    %timeit timematmul(A,B)

print("A sharding  P('data', None):")
jax.debug.visualize_array_sharding(A)
print("\nB sharding  P(None, 'model'):")
jax.debug.visualize_array_sharding(B)
print("\nC = A @ B inferred sharding:")
jax.debug.visualize_array_sharding(C)
print()
report(jax.jit(matmul), A, B, label="no-comm matmul")

print("\n\n===============================================================\n\n")

A = jax.device_put(A_h, jax.P("data", "model"))
B = jax.device_put(B_h, jax.P("model", None))
C = matmul(A, B)

jax.block_until_ready(C)

if timing_flag:
    %timeit timematmul(A,B)

print("A sharding  P('data', 'model'):")
jax.debug.visualize_array_sharding(A)
print("\nB sharding  P('model', None):")
jax.debug.visualize_array_sharding(B)
print("\nC = A @ B inferred sharding:")
jax.debug.visualize_array_sharding(C)
print()
report(jax.jit(matmul), A, B, label="comm matmul")

print("\n\n===============================================================\n\n")

A = jax.device_put(A_h, jax.P("data", "model"))
B = jax.device_put(B_h, jax.P("model", "data"))
C = matmul(A, B)

jax.block_until_ready(C)

if timing_flag:
    %timeit timematmul(A,B)

print("A sharding  P('data', 'model'):")
jax.debug.visualize_array_sharding(A)
print("\nB sharding  P('model', 'data'):")
jax.debug.visualize_array_sharding(B)
print("\nC = A @ B inferred sharding:")
jax.debug.visualize_array_sharding(C)
print()
report(jax.jit(matmul), A, B, label="even more comm matmul")

print("\n\n===============================================================\n\n")

A = jax.device_put(A_h, jax.P("data", "model"))
B = jax.device_put(B_h, jax.P("model", "data"))

@jax.jit(out_shardings=jax.P("data", "model")) # can also use jax.lax.with_sharding_constraint
def matmul(a, b):
    return a @ b

def timematmul(a,b):
    c = matmul(a,b)
    c = jax.block_until_ready(c)
    return c

C = matmul(A,B)
jax.block_until_ready(C)

if timing_flag:
    %timeit timematmul(A,B)

jax.debug.visualize_array_sharding(C)
report(jax.jit(matmul, out_shardings = jax.P("data", "model")), A, B, label="enforcing out_sharding")

# matmul is O(MNK) flops, i.e. cubic complexity. Memory movement is O(NK + KM) so it is of lower order (quadratic complexity).

A sharding  P('data', None):


                                                                           
                                                                           
                                 CPU 0,1,2                                 
                                                                           
                                                                           
                                                                           
                                                                           
                                                                           
                                 CPU 3,4,5                                 
                                                                           
                                                                           
                                                                           


B sharding  P(None, 'model'):


                           
                           
                           
                           
                           
 CPU 0,3  CPU 1,4  CPU 2,5 
                           
                           
                           
                           
                           


C = A @ B inferred sharding:


...


C = A @ B inferred sharding:


                                    
                                    
                                    
                                    
                                    
    CPU 0,1,2         CPU 3,4,5     
                                    
                                    
                                    
                                    
                                    


[even more comm matmul] collectives: 3
[even more comm matmul] types: all-gather=2, all-reduce=1






                                    
                                    
   CPU 0       CPU 1       CPU 2    
                                    
                                    
                                    
                                    
                                    
   CPU 3       CPU 4       CPU 5    
                                    
                                    
                                    

[enforcing out_sharding] collectives: 4
[enforcing out_sharding] types: all-gather=2, all-reduce=2


## Prallelism with sharding when training AI models

**Three main strategies for parallelism:**

* **Data parallel.** Split your data across different devices, i.e., shard along the batch size. Keep the same copy of the weights across all devices.
* **Model parallel.** Keep the whole dataset on all devices, but split the weights.
* **2D parallelism.** Do both.

**Note:** These are typical, but not fixed. If you think another strategy would work better, then go for it!

## Question 2 - Sharding parallelism

### a) Data parallelism

Implement the forward pass (i.e., the evaluation) of a single 2-layer multi-layer perceptron (MLP)

$$y = W_1\, \mathrm{relu}(W_0 x)$$

with data parallelism on a 1D `(6,)` mesh with label `'data'`:

- $x$ of shape $(n_\mathrm{in}, b)$ is sharded along the batch dimension $b$.
- $W_0$ of shape $(n_\mathrm{hidden}, n_\mathrm{in})$ is *replicated* on every device.
- $W_1$ of shape $(n_\mathrm{out}, n_\mathrm{hidden})$ is also *replicated*.

Set $n_\mathrm{in}=6$, $\ n_\mathrm{hidden}=12$, $\ n_\mathrm{out}=6$, $\ b = 30$,    and choose the right `PartitionSpec`s to reflect the above. After the forward pass, verify:

1. $y$ is sharded along the batch dimension (same as $x$).
2. **Zero collectives** appear when using `report` on the resulting MLP.

### b) Model parallelism

Now, shard the weights along `"model"` instead of the batch. Create a new 1D `(6,)` mesh with label `"model"`. Then take:

- $x$ of shape $(n_\mathrm{in}, b)$: *replicated*.
- $W_0$ of shape $(n_\mathrm{hidden}, n_\mathrm{in})$.
- $W_1$ of shape $(n_\mathrm{out}, n_\mathrm{hidden})$.

How should $W_0$ and $W_1$ be sharded to minimise communication? Hint: you have two options for each of them since you can shard rows or shard columns.

Verify the sharding of $y$ and use `report` to see how many collective communications were invoked. If you do things correctly, you should need exactly one all-reduce (after the second matmul), leading to a replicated output $y$ across all devices.

### c) 2D parallelism

Finally, implement the MLP forward pass by using a 2D parallelisation strategy.

Create a 2D mesh of axes `(3,2)` with labels `"model"`, `"data"`. Then take:

- $x$ of shape $(n_\mathrm{in}, b)$: split along the `"data"` axis.
- $W_0$ of shape $(n_\mathrm{hidden}, n_\mathrm{in})$: split along the `"model"` axis (by row or by column?).
- $W_1$ of shape $(n_\mathrm{out}, n_\mathrm{hidden})$: split along the `"model"` axis (by row or by column?).

Choose the shardings to minimise communication. Use `report` to see how many collective communications were invoked. Again you should be able to obtain a forward pass that only requires a single all-reduce. How do you expect the output to be sharded? Verify it.

## Solution

In [3]:
print("\n\n============ Part a) =============\n\n")

Auto = jax.sharding.AxisType.Auto
mesh = jax.make_mesh((6,), ("data",), (Auto,), devices=cpus)
jax.set_mesh(mesh)

n_in = 6
n_hidden = 12
n_out = 4
b = 30

key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

# x has shape (n_in, b), batch dimension is axis 1
x  = jax.random.normal(k1, (n_in, b))
W0 = jax.random.normal(k2, (n_hidden, n_in))
W1 = jax.random.normal(k3, (n_out, n_hidden))

# Data parallel sharding:
# - x sharded along batch axis b
# - W0, W1 replicated
x_s  = jax.device_put(x,  jax.P(None, "data"))
W0_s = jax.device_put(W0, jax.P())   # replicated
W1_s = jax.device_put(W1, jax.P())   # replicated

@jax.jit
def mlp(x, W0, W1):
    return W1 @ jax.nn.relu(W0 @ x)

y = mlp(x_s, W0_s, W1_s)

print("y sharding (batch sharded along 'data'):")
jax.debug.visualize_array_sharding(y)
print()

report(jax.jit(mlp), x_s, W0_s, W1_s, label="data-parallel MLP")

print("\n\n============ Part b) =============\n\n")

mesh = jax.make_mesh((6,), ("model",), (Auto,), devices=cpus)
jax.set_mesh(mesh)

# Model parallel sharding:
# - x replicated
# - W0, W1 sharded across the "model" axis
x_s  = jax.device_put(x,  jax.P())
W0_s = jax.device_put(W0, jax.P("model", None))   # row parallel
W1_s = jax.device_put(W1, jax.P(None, "model"))   # column parallel

y = mlp(x_s, W0_s, W1_s)

print("y sharding (replicated after the all-reduce):")
jax.debug.visualize_array_sharding(y)
print()

report(jax.jit(mlp), x_s, W0_s, W1_s, label="model-parallel MLP")

print("\n\n============ Part c) =============\n\n")

mesh = jax.make_mesh((3,2), ("model","data"), (Auto,Auto), devices=cpus)
jax.set_mesh(mesh)

# 2D parallel sharding:
# - x data sharded
# - W0, W1 model sharded
x_s  = jax.device_put(x,  jax.P(None, "data"))
W0_s = jax.device_put(W0, jax.P("model", None))   # row parallel
W1_s = jax.device_put(W1, jax.P(None, "model"))   # column parallel

y = mlp(x_s, W0_s, W1_s)

print("y sharding (data sharded):")
jax.debug.visualize_array_sharding(y)
print()

report(jax.jit(mlp), x_s, W0_s, W1_s, label="2D-parallel MLP")



============ Part a) =============


y sharding (batch sharded along 'data'):



[data-parallel MLP] collectives: 0
[data-parallel MLP] types: none


============ Part b) =============


y sharding (replicated after the all-reduce):


                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                CPU 0,1,2,3,4,5                                 
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                


[model-parallel MLP] collectives: 1
[model-parallel MLP] types: all-reduce=1


============ Part c) =============


y sharding (data sharded):



[2D-parallel MLP] collectives: 1
[2D-parallel MLP] types: all-reduce=1


### Question 3 - Does it pay off?

We will now explore whether sharding can actually accelerate our computations.

Take the same MLP as in the previous question. Consider all parallelism strategies a), b), c) and also implement a fully replicated MLP (in which you use `jax.P()` on a 1D mesh so that all arrays are replicated across devices and no parallelisation occurs).

Using the auxiliary function `time_fn` (defined in the first code block of this practical) with usage

```python
rep_med = time_fn(mlp, x, W0, W1) # rep_med is the median time it took to call mlp(x,W0,w1) in seconds  
```

Check which implementation strategy is faster for increasing problem sizes: take $n_\mathrm{in}=12$ and $n_\mathrm{out}=4$, but then use increasing $n_\mathrm{hidden} \in \{1024, 4096, 8184\}$ and $b\in \{258, 1026, 2046\}$.

Are the results as expected? Why is the no parallelism option slower? Do you think there are any additional benefits of using sharding beyond increasing the speed of computations?

## Solution

In [4]:
Auto = jax.sharding.AxisType.Auto

n_in = 12
n_out = 4

key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

hidden_sizes = [4*258, 4*1026, 4*2046]
batch_sizes  = [258, 1026, 2046]

assert len(hidden_sizes) == len(batch_sizes)

times = np.zeros((len(hidden_sizes), 4))

for i,(n_hidden,b) in enumerate(zip(hidden_sizes, batch_sizes)):
    
    # x has shape (n_in, b), batch dimension is axis 1
    x  = jax.random.normal(k1, (n_in, b))
    W0 = jax.random.normal(k2, (n_hidden, n_in))
    W1 = jax.random.normal(k3, (n_out, n_hidden))

    #print("\n\n============ Data parallel =============\n\n")

    mesh = jax.make_mesh((6,), ("data",), (Auto,), devices=cpus)
    jax.set_mesh(mesh)

    # Data parallel sharding:
    # - x sharded along batch axis b
    # - W0, W1 replicated
    x_s  = jax.device_put(x,  jax.P(None, "data"))
    W0_s = jax.device_put(W0, jax.P())   # replicated
    W1_s = jax.device_put(W1, jax.P())   # replicated

    @jax.jit
    def mlp(x, W0, W1):
        return W1 @ jax.nn.relu(W0 @ x)

    y = mlp(x_s, W0_s, W1_s)
    y = jax.block_until_ready(y)

    t = time_fn(mlp, x_s, W0_s, W1_s)
    times[i][0] = t

    #print("\n\n============ Model parallel =============\n\n")

    mesh = jax.make_mesh((6,), ("model",), (Auto,), devices=cpus)
    jax.set_mesh(mesh)

    # Model parallel sharding:
    # - x replicated
    # - W0, W1 sharded across the "model" axis
    x_s  = jax.device_put(x,  jax.P())
    W0_s = jax.device_put(W0, jax.P("model", None))   # row parallel
    W1_s = jax.device_put(W1, jax.P(None, "model"))   # column parallel

    @jax.jit
    def mlp(x, W0, W1):
        return W1 @ jax.nn.relu(W0 @ x)

    y = mlp(x_s, W0_s, W1_s)
    y = jax.block_until_ready(y)

    t = time_fn(mlp, x_s, W0_s, W1_s)
    times[i][1] = t

    #print("\n\n============ 2D parallelism =============\n\n")

    mesh = jax.make_mesh((3,2), ("model","data"), (Auto,Auto), devices=cpus)
    jax.set_mesh(mesh)

    # 2D parallel sharding:
    # - x data sharded
    # - W0, W1 model sharded
    x_s  = jax.device_put(x,  jax.P(None, "data"))
    W0_s = jax.device_put(W0, jax.P("model", None))   # row parallel
    W1_s = jax.device_put(W1, jax.P(None, "model"))   # column parallel

    @jax.jit
    def mlp(x, W0, W1):
        return W1 @ jax.nn.relu(W0 @ x)

    y = mlp(x_s, W0_s, W1_s)
    y = jax.block_until_ready(y)


    t = time_fn(mlp, x_s, W0_s, W1_s)
    times[i][2] = t

    #print("\n\n============ Full replication - no parallelism =============\n\n")

    mesh = jax.make_mesh((6,), ("X",), (Auto,), devices=cpus)
    jax.set_mesh(mesh)

    # full replication, i.e., no parallelism
    x_s  = jax.device_put(x,  jax.P())
    W0_s = jax.device_put(W0, jax.P())
    W1_s = jax.device_put(W1, jax.P())

    @jax.jit
    def mlp(x, W0, W1):
        return W1 @ jax.nn.relu(W0 @ x)

    y = mlp(x_s, W0_s, W1_s)
    y = jax.block_until_ready(y)

    t = time_fn(mlp, x_s, W0_s, W1_s)
    times[i][3] = t

labels = ["Data", "Model", "2D"]

baseline = times[:, -1][:, None]
speedups = baseline / times
speedups_par = speedups[:, :-1]

print("\nSpeedups (relative to 'no parallelism'):\n")

header = "".join(f"{l:>10}" for l in labels)
print(header)
for row in speedups_par:
    print("".join(f"{val:10.2f}" for val in row))

# By using sharding the amount of per-device memory required decreases 


Speedups (relative to 'no parallelism'):

      Data     Model        2D
      2.13      2.50      1.93
     16.54     16.28     16.20
      6.16      6.50      6.60


## Question 4 - Resharding

Up to now, JAX's compiler picks a sharding for every intermediate value automatically, and propagation "just works". But the locally-optimal pick at each operation may not be globally optimal across the whole computation. That's where **re-sharding** matters.

Re-sharding can be:

1. **Necessary** — two computations have incompatible sharded memory layouts. JAX inserts an `all-to-all` or `all-gather` to bridge them.
2. **Beneficial** — changing sharding makes the computation cheaper.
3. **Wasteful** — unnecessary resharding that makes computations slower.

This exercise showcases case 3.

### a) The cost of unnecessary resharding

Let $n=2046$, $b=60$ and take $x$ to be a matrix of size $(n, b)$ and define a function that computes a chain of $4$ matrix multiplications $y = W_3W_2W_1W_0x$ (you can add activation functions if you want) in alternating sharding pattern (`W0` row-parallel, `W1` column-parallel, `W2` row-parallel, `W3` column-parallel) using a 1D mesh of $6$ CPUs. Take the matrices $W_i$ to be of size $n$-by-$n$ for all $i$ and fully replicate $x$ using `jax.P()`.

Compare two implementations:

- **Automatic**: let JAX figure out the intermediate shardings.
- **Over-constrained**: insert `jax.lax.with_sharding_constraint(..., P())` after every matrix multiplication to force fully replicated shardings.

The over-constrained version forces an extra all-gather between every pair of layers — collectives that the propagated version avoids. Measure the difference in collective count and wall time using `time_fn` and `report`.

### b) Explicit axes

This part is only so that you get familiar with `Explicit` axes (the JAX default) and so that you see that you have much more control with it.

Now, repeat the excercise using `Explicit` axes, but without using the decorator `@auto_axes` (you also won't need `jax.lax.with_sharding_constraint`). Try to reproduce what JAX does automatically, check with `report` that you are doing it correctly. **Recall:** You will have to use the `out_sharding=` keyword in `jax.numpy.dot` and in `jax.jit`.

## Solution

In [5]:
Auto = jax.sharding.AxisType.Auto
Explicit = jax.sharding.AxisType.Explicit

b = 60
N = 2046
keys = jax.random.split(jax.random.PRNGKey(0), 5)
xc = jax.random.normal(keys[0], (N,b))
Wc1, Wc2, Wc3, Wc4 = (jax.random.normal(k, (N, N)) for k in keys[1:])

print("\n################ Part a) #####################\n")

mesh = jax.make_mesh((6,), ("model",), (Auto,), devices=cpus)
jax.set_mesh(mesh)

xc_r  = jax.device_put(xc,  jax.P())
Wc1_m = jax.device_put(Wc1, jax.P("model", None))
Wc2_m = jax.device_put(Wc2, jax.P(None, "model"))
Wc3_m = jax.device_put(Wc3, jax.P("model", None))
Wc4_m = jax.device_put(Wc4, jax.P(None, "model"))

@jax.jit
def automatic(x, W1, W2, W3, W4):
    h = W1@x
    h = W2@h
    h = W3@h
    h = W4@h
    return h

@jax.jit
def overconstrained(x, W1, W2, W3, W4):
    h = W1@x
    h = jax.lax.with_sharding_constraint(h, jax.P())
    h = W2@h
    h = jax.lax.with_sharding_constraint(h, jax.P())
    h = W3@h
    h = jax.lax.with_sharding_constraint(h, jax.P())
    h = W4@h
    h = jax.lax.with_sharding_constraint(h, jax.P())
    return h

for s,fn in zip(["automatic", "overconstrained"],[automatic, overconstrained]):
    print("\n" + "-"*10 + "\n\nSharding strategy: " + s + ".\n")
    report(jax.jit(fn), xc_r, Wc1_m, Wc2_m, Wc3_m, Wc4_m)
    t = time_fn(fn, xc_r, Wc1_m, Wc2_m, Wc3_m, Wc4_m)
    print("\nComputational time: %.3f ms" % t)

print("\n################ Part b) #####################\n")

mesh = jax.make_mesh((6,), ("model",), (Explicit,), devices=cpus)
jax.set_mesh(mesh)

xc_r  = jax.device_put(xc,  jax.P())
Wc1_m = jax.device_put(Wc1, jax.P("model", None))
Wc2_m = jax.device_put(Wc2, jax.P(None, "model"))
Wc3_m = jax.device_put(Wc3, jax.P("model", None))
Wc4_m = jax.device_put(Wc4, jax.P(None, "model"))

@jax.jit
def automatic(x, W1, W2, W3, W4):
    h = jnp.dot(W1, x, out_sharding=jax.P("model", None))
    h = jnp.dot(W2, h, out_sharding=jax.P())
    h = jnp.dot(W3, h, out_sharding=jax.P("model", None))
    h = jnp.dot(W4, h, out_sharding=jax.P())
    return h

@jax.jit
def overconstrained(x, W1, W2, W3, W4):
    h = jnp.dot(W1, x, out_sharding=jax.P())
    h = jnp.dot(W2, h, out_sharding=jax.P())
    h = jnp.dot(W3, h, out_sharding=jax.P())
    h = jnp.dot(W4, h, out_sharding=jax.P())
    return h

for s,fn in zip(["automatic", "overconstrained"],[automatic, overconstrained]):
    print("\n" + "-"*10 + "\n\nSharding strategy: " + s + ".\n")
    report(jax.jit(fn), xc_r, Wc1_m, Wc2_m, Wc3_m, Wc4_m)
    t = time_fn(fn, xc_r, Wc1_m, Wc2_m, Wc3_m, Wc4_m)
    print("\nComputational time: %.3f ms" % t)


################ Part a) #####################


----------

Sharding strategy: automatic.

collectives: 3
types: all-reduce=3

Computational time: 5.332 ms

----------

Sharding strategy: overconstrained.

collectives: 6
types: all-gather=3, all-reduce=3

Computational time: 74.167 ms

################ Part b) #####################


----------

Sharding strategy: automatic.

collectives: 3
types: all-reduce=3

Computational time: 8.927 ms

----------

Sharding strategy: overconstrained.

collectives: 6
types: all-gather=3, all-reduce=3

Computational time: 66.747 ms
